So this is where i'm gonna start working on the educational version. I'd probably want a background on both peft and bayesian techniques here

In [17]:
pip install datasets transformers peft evaluate torchmetrics scikit-learn, ipdb

Note: you may need to restart the kernel to use updated packages.


ERROR: Invalid requirement: 'scikit-learn,': Expected end or semicolon (after name and no valid version specifier)
    scikit-learn,
                ^


In [18]:
from datasets import load_dataset

#Download the Rotten Tomatoes dataset directly from hugging face
raw_dataset = load_dataset("rotten_tomatoes")

#shrink it
#The original training set has 8,500 reviews. We will shuffle them and grab exactly 1,000.
#We will also grab 200 for your testing/evaluation set.
#1000 doesn't seem like a lot, but for deep learning on an unimpressive cpu, this is already a lot.
small_train_dataset = raw_dataset["train"].shuffle(seed=42).select(range(1000))
small_test_dataset = raw_dataset["test"].shuffle(seed=42).select(range(200))

print("\n--- Dataset Ready! ---") #yay!
print(f"Training examples: {len(small_train_dataset)}")
print(f"Testing examples: {len(small_test_dataset)}\n")

#A look at the very first example to see what we are working with
print(f"Text: '{small_train_dataset[0]['text']}'")
print(f"Label: {small_train_dataset[0]['label']} (0 = Negative, 1 = Positive)")


--- Dataset Ready! ---
Training examples: 1000
Testing examples: 200

Text: '. . . plays like somebody spliced random moments of a chris rock routine into what is otherwise a cliche-riddled but self-serious spy thriller .'
Label: 0 (0 = Negative, 1 = Positive)


In [19]:
import torch
#import ipdb
from modelwrappers.wrapperbase import WrapperBase

class EducationalBayesianWrapper(WrapperBase):
    def __init__(self, model, peft_config, args, accelerator, adapter_name="default"):
        #Initialize the professor's base class (handles the optimizer, metrics, etc.)
        super().__init__(model, peft_config, args, accelerator, adapter_name)

    def forward_logits(self, batch, sample=False, n_samples=1):
        """
        The Educational Engine:
        Takes a batch of text, passes it through RoBERTa, and returns the logits.
        """
        #Extract the text tokens and attention masks from the trimmed review batch
        input_ids = batch['input_ids']
        attention_mask = batch['attention_mask']

        #standard pass through the model without sampling (just one guess)
        if not sample or n_samples == 1:
            outputs = self.base_model(
                input_ids=input_ids, 
                attention_mask=attention_mask
            )
            #The professor's evaluation math expects a 3D tensor: [batch_size, n_samples, classes]
            #So we add a dimension in the middle with unsqueeze(1)
            return outputs.logits.unsqueeze(1)

        #Bayesian pass
        else:
            #force the model into training mode to activate the random dropout layers
            self.base_model.train() 
            
            stacked_logits = []
            
            #Run the exact same text through the model n_samples times (e.g., 10 times)
            for _ in range(n_samples):
                outputs = self.base_model(
                    input_ids=input_ids, 
                    attention_mask=attention_mask
                )
                stacked_logits.append(outputs.logits)
            
            #safely return the model to evaluation mode
            self.base_model.eval()

            #stack all 10 guesses together into a single block of math
            #Final Shape: [batch_size, 10, 2]
            return torch.stack(stacked_logits, dim=1)

now we wanna tokenize the data twin!!!!

In [20]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model
from torch.utils.data import DataLoader
from accelerate import Accelerator


#tokenize the data. English into matrix 
model_name = "distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenizing(examples):
    #remember that our goal is to be able to run this on a normal computer, which is why max_length is set to 128 (instead of 512 for the full roberta)
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

#tokenize the training and testing sets
tokenized_train = small_train_dataset.map(tokenizing, batched=True)
tokenized_test = small_test_dataset.map(tokenizing, batched=True)

#then convert to pytorch tensors for the model
tokenized_train.set_format("torch", columns=["input_ids", "attention_mask", "label"])
tokenized_test.set_format("torch", columns=["input_ids", "attention_mask", "label"])

#then group the data into small manageable batches for training and evaluation
train_loader = DataLoader(tokenized_train, batch_size=16, shuffle=True)
test_loader = DataLoader(tokenized_test, batch_size=16)

#now we wanna build the base model
#load the model first
#only two labels, positive and negative
base_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

#parameter efficient adapter configuration
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules = ["query", "value"],
    lora_dropout=0.1
)

#put adapter on base model
peft_model = get_peft_model(base_model, lora_config)


#initializing OUR wrapper
#prof's base class expects a huge args file. We're running ts on performance mode at 12 fps
#aka lowest settings. Note we're doing this because we are in a jupyter notebook
class NotebookArgs:
    batch_size = 16
    n_epochs = 1
    max_train_steps = 0
    outdim = 2
    opt = "adamw"
    lr = 1e-4
    opt_wd = 0.01
    adam_epsilon = 1e-8
    warmup_ratio = 0.1
    dataset_type = "bertds" # The "secret backdoor" we found earlier!
    epoch = 0
    eval_per_steps = 1000
    num_samples = 1000

accelerator = Accelerator() # Automatically routes math to your CPU

bayesian_model = EducationalBayesianWrapper(
    model=peft_model,
    peft_config=lora_config,
    args=NotebookArgs(),
    accelerator=accelerator
)

#trying it
print("\nFiring up the Bayesian Engine...")
# Grab exactly one batch of 16 reviews
test_batch = next(iter(train_loader)) 

# Push it through the Bayesian pass you just coded (asking for 5 samples)
bayesian_logits = bayesian_model.forward_logits(test_batch, sample=True, n_samples=5)

print("--- Engine Test Complete ---")
print(f"Success! Output tensor shape is: {bayesian_logits.shape}")
print("(It should read: [16, 5, 2] -> 16 reviews, 5 guesses each, 2 possible labels)")


Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
c:\Users\hsing\anaconda3\Lib\site-packages\peft\tuners\tuners_utils.py:285: UserWarning: Already found a `peft


Firing up the Bayesian Engine...
--- Engine Test Complete ---
Success! Output tensor shape is: torch.Size([16, 5, 2])
(It should read: [16, 5, 2] -> 16 reviews, 5 guesses each, 2 possible labels)
